# Resolução — S03-03

Notebook em **Julia**, corrigido para atender aos requisitos pedido pelo professor 
Prof. Kenny Vinente dos Santos, Dr. | kennyvinente@ufam.edu.br
# Nome: Renyer Montefusco levy
# Matrícula: 2240917

Resolução feita em **Julia**, seguindo os itens solicitados no notebook original do professor.


### Pacotes importados

In [1]:
using LinearAlgebra
using Printf

## Chapter 9: Quadratic problems

Consideramos o problema quadrático:

\[
f(x)=\frac{1}{2}x^TQx+b^Tx
\]

Quando \(Q\) é simétrica definida positiva, o ponto minimizador satisfaz:

\[
\nabla f(x)=Qx+b=0
\]

Logo:

\[
x^*=-Q^{-1}b
\]

Na prática, resolvemos o sistema linear:

\[
Qx=-b
\]


# Algorithm 9.1 — Quadratic problems: direct solution

Exemplo 9.8:

\[
Q=
\begin{bmatrix}
1 & 1 & 1 & 1 \\
1 & 2 & 2 & 2 \\
1 & 2 & 3 & 3 \\
1 & 2 & 3 & 4
\end{bmatrix}
\]

\[
b=
\begin{bmatrix}
-4 \\
-7 \\
-9 \\
-10
\end{bmatrix}
\]

Primeiro, confirmamos que \(Q\) é definida positiva.


In [2]:
Q = [
    1.0  1.0  1.0  1.0;
    1.0  2.0  2.0  2.0;
    1.0  2.0  3.0  3.0;
    1.0  2.0  3.0  4.0
]

b = [-4.0, -7.0, -9.0, -10.0]

println("Matriz Q:")
display(Q)

println("Vetor b:")
display(b)

println("Q é simétrica? ", issymmetric(Q))

autovalores = eigvals(Q)
println("Autovalores de Q:")
display(autovalores)

if all(autovalores .> 0)
    println("Conclusão: Q é definida positiva.")
else
    println("Conclusão: Q NÃO é definida positiva.")
end

Matriz Q:


4×4 Matrix{Float64}:
 1.0  1.0  1.0  1.0
 1.0  2.0  2.0  2.0
 1.0  2.0  3.0  3.0
 1.0  2.0  3.0  4.0

Vetor b:


4-element Vector{Float64}:
  -4.0
  -7.0
  -9.0
 -10.0

Q é simétrica? true
Autovalores de Q:


4-element Vector{Float64}:
 0.2831185828579487
 0.42602204776046276
 1.0
 8.290859369381593

Conclusão: Q é definida positiva.


## Solução direta

Como \(Q\) é definida positiva, a solução direta é:

\[
x^* = -Q^{-1}b
\]

ou, computacionalmente:

\[
Qx=-b
\]


In [3]:
function quadratic_direct_solution(Q, b)
    if !issymmetric(Q)
        error("A matriz Q não é simétrica.")
    end

    autovalores = eigvals(Q)

    if !all(autovalores .> 0)
        error("A matriz Q não é definida positiva.")
    end

    x = Q \ (-b)
    return x
end

x_direto = quadratic_direct_solution(Q, b)

println("Solução direta x*:")
display(x_direto)

gradiente = Q*x_direto + b

println("Gradiente Qx + b:")
display(gradiente)

println("Norma do gradiente = ", norm(gradiente))

valor_objetivo = 0.5 * dot(x_direto, Q*x_direto) + dot(b, x_direto)
println("Valor da função objetivo f(x*) = ", valor_objetivo)

Solução direta x*:


4-element Vector{Float64}:
 1.0
 1.0
 1.0
 1.0

Gradiente Qx + b:


4-element Vector{Float64}:
 0.0
 0.0
 0.0
 0.0

Norma do gradiente = 0.0
Valor da função objetivo f(x*) = -15.0


# Algorithm 9.2 — Conjugate Gradient Method

Agora aplicamos o método do gradiente conjugado para resolver o mesmo problema quadrático.

O método parte de:

\[
x_0=
\begin{bmatrix}
5 \\
5 \\
5 \\
5
\end{bmatrix}
\]

Como:

\[
\nabla f(x)=Qx+b
\]

usamos:

\[
g_k=Qx_k+b
\]

e atualizamos as direções conjugadas.


In [4]:
function conjugate_gradient_quadratic(Q, b, x0; eps=1e-12, maxiter=100)
    if !issymmetric(Q)
        error("A matriz Q não é simétrica.")
    end

    autovalores = eigvals(Q)

    if !all(autovalores .> 0)
        error("A matriz Q não é definida positiva.")
    end

    x = float.(x0)
    g = Q*x + b
    d = -g

    historico = []

    for k in 0:maxiter
        push!(historico, (
            Iteracao=k,
            x=copy(x),
            gradiente=copy(g),
            norma_gradiente=norm(g)
        ))

        if norm(g) <= eps
            break
        end

        alpha = dot(g, g) / dot(d, Q*d)

        x_novo = x + alpha*d
        g_novo = Q*x_novo + b

        beta = dot(g_novo, g_novo) / dot(g, g)

        d = -g_novo + beta*d
        x = x_novo
        g = g_novo
    end

    return x, historico
end

function mostrar_historico_cg(historico)
    println("Iteração | x | ||gradiente||")
    println("--------------------------------------------------------------------------")

    for linha in historico
        @printf("%8d | [% .8f, % .8f, % .8f, % .8f] | %.16e\n",
            linha.Iteracao,
            linha.x[1],
            linha.x[2],
            linha.x[3],
            linha.x[4],
            linha.norma_gradiente
        )
    end
end

mostrar_historico_cg (generic function with 1 method)

## Execução do Gradiente Conjugado

Executando o algoritmo com:

\[
x_0=
\begin{bmatrix}
5 \\
5 \\
5 \\
5
\end{bmatrix}
\]


In [5]:
x0 = [5.0, 5.0, 5.0, 5.0]

x_cg, historico_cg = conjugate_gradient_quadratic(Q, b, x0)

mostrar_historico_cg(historico_cg)

println()
println("Solução pelo gradiente conjugado:")
display(x_cg)

println("Gradiente final Qx + b:")
display(Q*x_cg + b)

println("Norma do gradiente final = ", norm(Q*x_cg + b))

println()
println("Comparação com a solução direta:")
display(x_direto)

println("Diferença entre solução direta e CG = ", norm(x_direto - x_cg))

Iteração | x | ||gradiente||
--------------------------------------------------------------------------
       0 | [ 5.00000000,  5.00000000,  5.00000000,  5.00000000] | 6.2737548565432490e+01
       1 | [ 3.06774669,  1.61855670,  0.65243004,  0.16936672] | 2.0859355510619855e+00
       2 | [ 1.49689643,  0.61022416,  0.84799284,  1.21554224] | 2.7758532578478146e-01
       3 | [ 1.02806438,  0.93809327,  1.07428807,  0.96533223] | 3.1202737385779476e-02
       4 | [ 1.00000000,  1.00000000,  1.00000000,  1.00000000] | 7.2949006924086549e-12
       5 | [ 1.00000000,  1.00000000,  1.00000000,  1.00000000] | 2.0350724194510405e-15

Solução pelo gradiente conjugado:


4-element Vector{Float64}:
 0.9999999999999992
 0.9999999999999998
 0.9999999999999991
 1.0000000000000013

Gradiente final Qx + b:


4-element Vector{Float64}:
 -4.440892098500626e-16
 -8.881784197001252e-16
  0.0
  1.7763568394002505e-15

Norma do gradiente final = 2.0350724194510405e-15

Comparação com a solução direta:


4-element Vector{Float64}:
 1.0
 1.0
 1.0
 1.0

Diferença entre solução direta e CG = 1.793620188084545e-15


# Teste de erro para matriz não definida positiva

O professor pede que um erro seja disparado quando a matriz não for definida positiva.

Agora testamos:

\[
Q=
\begin{bmatrix}
1 & 2 & 3 & 4 \\
5 & 6 & 7 & 8 \\
9 & 10 & 11 & 12 \\
13 & 14 & 15 & 16
\end{bmatrix}
\]


In [6]:
Q_nao_pd = [
    1.0   2.0   3.0   4.0;
    5.0   6.0   7.0   8.0;
    9.0   10.0  11.0  12.0;
    13.0  14.0  15.0  16.0
]

println("Matriz Q não definida positiva:")
display(Q_nao_pd)

println("Q é simétrica? ", issymmetric(Q_nao_pd))

println("Tentando aplicar a solução direta:")

try
    quadratic_direct_solution(Q_nao_pd, b)
catch erro
    println("Erro capturado corretamente:")
    println(erro)
end

println()
println("Tentando aplicar o gradiente conjugado:")

try
    conjugate_gradient_quadratic(Q_nao_pd, b, x0)
catch erro
    println("Erro capturado corretamente:")
    println(erro)
end

Matriz Q não definida positiva:


4×4 Matrix{Float64}:
  1.0   2.0   3.0   4.0
  5.0   6.0   7.0   8.0
  9.0  10.0  11.0  12.0
 13.0  14.0  15.0  16.0

Q é simétrica? false
Tentando aplicar a solução direta:
Erro capturado corretamente:
ErrorException("A matriz Q não é simétrica.")

Tentando aplicar o gradiente conjugado:
Erro capturado corretamente:
ErrorException("A matriz Q não é simétrica.")
